# Anime Recommendation System (Cosine Similarity)


## 1. Data Preprocessing

In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\Admin\OneDrive\Desktop\Data Science ExcelR\Assignments\Assignment 16\anime.csv")
print(df.shape)
df.head()

(12294, 7)


,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
# Explore structure
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [6]:
# Handle missing values
df['genre'] = df['genre'].fillna('Unknown')
df['type'] = df['type'].fillna('Unknown')
df['rating'] = df['rating'].fillna(df['rating'].mean())
df['episodes'] = pd.to_numeric(df['episodes'], errors='coerce')
df['episodes'] = df['episodes'].fillna(df['episodes'].median())

df = df.drop_duplicates(subset='name').reset_index(drop=True)
df.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

## 2. Feature Extraction

In [7]:
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

# Genres -> multi-hot encoding
genre_lists = df['genre'].apply(lambda x: [g.strip() for g in x.split(',')])
mlb = MultiLabelBinarizer()
genre_features = mlb.fit_transform(genre_lists)

# Type -> one-hot encoding
type_features = pd.get_dummies(df['type']).values

# Numeric features -> normalized
scaler = MinMaxScaler()
numeric_features = scaler.fit_transform(df[['rating', 'episodes', 'members']])

# Combine all features into one matrix
feature_matrix = np.hstack([genre_features, type_features, numeric_features])
feature_matrix.shape

(12292, 54)

## 3. Recommendation System

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(feature_matrix)
title_to_idx = pd.Series(df.index, index=df['name'])

def recommend(anime_name, top_n=10, threshold=0.0):
    """Recommend anime similar to `anime_name` using cosine similarity."""
    if anime_name not in title_to_idx:
        return f"'{anime_name}' not found in dataset."

    idx = title_to_idx[anime_name]
    scores = list(enumerate(similarity_matrix[idx]))
    scores = [s for s in scores if s[0] != idx and s[1] >= threshold]
    scores.sort(key=lambda x: x[1], reverse=True)
    scores = scores[:top_n]

    result = df.iloc[[i for i, _ in scores]][['name', 'genre', 'type', 'rating']].copy()
    result['similarity'] = [round(s, 3) for _, s in scores]
    return result.reset_index(drop=True)

recommend('Naruto', top_n=10)

,name,genre,type,rating,similarity
0,Naruto: Shippuuden,"Action, Comedy, Martial Arts, Shounen, Super P...",TV,7.94,0.997
1,Katekyo Hitman Reborn!,"Action, Comedy, Shounen, Super Power",TV,8.37,0.912
2,Dragon Ball Z,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,8.32,0.873
3,Bleach,"Action, Comedy, Shounen, Super Power, Supernat...",TV,7.95,0.856
4,Dragon Ball Kai,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,7.95,0.856
5,Dragon Ball Super,"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,7.40,0.853
6,Medaka Box,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.21,0.853
7,Tenjou Tenge,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.10,0.852
8,Medaka Box Abnormal,"Action, Comedy, Ecchi, Martial Arts, School, S...",TV,7.63,0.851
9,Dragon Ball Kai (2014),"Action, Adventure, Comedy, Fantasy, Martial Ar...",TV,8.01,0.850


In [9]:
# Experiment with different thresholds
for t in [0.0, 0.3, 0.5, 0.7]:
    n = len(recommend('Naruto', top_n=50, threshold=t))
    print(f"threshold={t}: {n} recommendations")

threshold=0.0: 50 recommendations
threshold=0.3: 50 recommendations
threshold=0.5: 50 recommendations
threshold=0.7: 50 recommendations


## Interview Questions

**1. Difference between user-based and item-based collaborative filtering?**

- *User-based*: finds users similar to the target user (based on rating patterns) and recommends items those similar users liked.
- *Item-based*: finds items similar to items the target user already liked (based on how users rated them) and recommends those similar items.
- Item-based is generally more stable and scalable since item-item relationships change less often than user preferences.

**2. What is collaborative filtering, and how does it work?**

Collaborative filtering recommends items based on patterns in user-item interactions (ratings, clicks, purchases) rather than item content. It assumes users with similar past behavior will have similar future preferences. It works by building a user-item matrix, computing similarity (between users or items), and predicting/recommending items based on the ratings of similar users or the similarity of items already liked — without needing explicit item features.